In [21]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
 
from langgraph.checkpoint.sqlite import SqliteSaver

import sqlite3
model = "qwen3.5:9b"
 

llm = ChatOllama(
    model=model,
    temperature=0
)
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]


def chat_node(state:ChatState)->ChatState:
    message = state['messages']
   
    response = llm.invoke(message)
    
    return{
        "messages":[response]
    }
    
connection = sqlite3.connect(database="chatbot.db",check_same_thread=False)    
checkpoint = SqliteSaver(connection)
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

# define edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot = graph.compile(checkpoint)
threads = checkpoint.list(None)
print(threads)


<generator object SqliteSaver.list at 0x000001EA4D8FB2A0>


In [22]:
def get_all_threads() -> list[str]:
    thread_ids = set()

    for checkpoint_tuple in checkpoint.list(None):
        thread_id = checkpoint_tuple.config["configurable"].get("thread_id")

        if thread_id:
            thread_ids.add(thread_id)

    return sorted(thread_ids)

In [23]:
x = get_all_threads
x

<function __main__.get_all_threads() -> list[str]>